# Run 322 artifact probe

Draft notebook for `run_id=322b4485-557c-434b-b2e1-0776d15a515b`.

This notebook loads the pinned BTCUSDT `1h` price artifacts and the two signal matrices used by the run:

- `ma.dema` from `signals/1h/ma.dema/signals.i8.npy`
- `ma.ema` from `signals/1h/ma.ema/signals.i8.npy`

Pinned runtime identity:

- slot: `slot_a`
- generation: `1`
- asof_date: `2026-04-02`
- manifest_hash: `13df35a144a9706b6b40c949f71ddc5dd23d60da10bbda81c12e26f9676faaae`
- timeframe: `1h`
- symbol: `BTCUSDT`


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import yaml

RUN_ID = "322b4485-557c-434b-b2e1-0776d15a515b"
TIMEFRAME = "1h"
RUN_TIME_RANGE_START = datetime(2017, 10, 6, 22, 55, tzinfo=timezone.utc)
RUN_TIME_RANGE_END = datetime(2026, 3, 31, 22, 55, tzinfo=timezone.utc)
RUN_TIME_RANGE_START_MS = int(RUN_TIME_RANGE_START.timestamp() * 1000)
RUN_TIME_RANGE_END_MS = int(RUN_TIME_RANGE_END.timestamp() * 1000)

ARTIFACT_ROOT = Path("/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a")
PRICE_DIR = ARTIFACT_ROOT / "prices" / TIMEFRAME
DEMA_DIR = ARTIFACT_ROOT / "signals" / TIMEFRAME / "ma.dema"
EMA_DIR = ARTIFACT_ROOT / "signals" / TIMEFRAME / "ma.ema"

SLOT_MANIFEST_PATH = ARTIFACT_ROOT / "manifest.yaml"
PRICE_OPEN_TIME_PATH = PRICE_DIR / "open_time.i64.npy"
PRICE_CLOSE_TIME_PATH = PRICE_DIR / "close_time.i64.npy"
PRICE_OHLCV_PATH = PRICE_DIR / "ohlcv.f32.npy"
DEMA_MANIFEST_PATH = DEMA_DIR / "manifest.yaml"
EMA_MANIFEST_PATH = EMA_DIR / "manifest.yaml"
DEMA_SIGNAL_PATH = DEMA_DIR / "signals.i8.npy"
EMA_SIGNAL_PATH = EMA_DIR / "signals.i8.npy"

REQUEST_INDICATOR_GRIDS = [
    {
        "indicator_id": "ma.dema",
        "sources": ["close"],
        "window_range": [5, 200],
    },
    {
        "indicator_id": "ma.ema",
        "sources": ["high", "ohlc4"],
        "window_range": [5, 200],
    },
]

for path in [
    SLOT_MANIFEST_PATH,
    PRICE_OPEN_TIME_PATH,
    PRICE_CLOSE_TIME_PATH,
    PRICE_OHLCV_PATH,
    DEMA_MANIFEST_PATH,
    EMA_MANIFEST_PATH,
    DEMA_SIGNAL_PATH,
    EMA_SIGNAL_PATH,
]:
    print(f"{path}: exists={path.exists()}")


In [ ]:
slot_manifest = yaml.safe_load(SLOT_MANIFEST_PATH.read_text())
dema_manifest = yaml.safe_load(DEMA_MANIFEST_PATH.read_text())
ema_manifest = yaml.safe_load(EMA_MANIFEST_PATH.read_text())

prices_by_timeframe = {item["timeframe"]: item for item in slot_manifest["prices"]}
price_manifest = prices_by_timeframe[TIMEFRAME]

print("slot:", slot_manifest["slot"], "generation:", slot_manifest["slot_generation"], "asof_date:", slot_manifest["asof_date"])
print("price bars:", price_manifest["coverage"]["bar_count"])
print("dema signals shape:", tuple(dema_manifest["signals"]["shape"]))
print("ema signals shape:", tuple(ema_manifest["signals"]["shape"]))
print("request indicator grids:", REQUEST_INDICATOR_GRIDS)


In [ ]:
price_open_time = np.load(PRICE_OPEN_TIME_PATH, mmap_mode="r")
price_close_time = np.load(PRICE_CLOSE_TIME_PATH, mmap_mode="r")
price_ohlcv = np.load(PRICE_OHLCV_PATH, mmap_mode="r")
dema_signals = np.load(DEMA_SIGNAL_PATH, mmap_mode="r")
ema_signals = np.load(EMA_SIGNAL_PATH, mmap_mode="r")

print("price_open_time:", price_open_time.shape, price_open_time.dtype)
print("price_close_time:", price_close_time.shape, price_close_time.dtype)
print("price_ohlcv:", price_ohlcv.shape, price_ohlcv.dtype)
print("dema_signals:", dema_signals.shape, dema_signals.dtype)
print("ema_signals:", ema_signals.shape, ema_signals.dtype)


In [ ]:
open_time_index = price_open_time.astype("datetime64[ms]")
close_time_index = price_close_time.astype("datetime64[ms]")
time_mask = (price_open_time >= RUN_TIME_RANGE_START_MS) & (price_open_time <= RUN_TIME_RANGE_END_MS)

print("bars inside run time range:", int(time_mask.sum()))
print("first matching bar:", np.datetime_as_string(open_time_index[time_mask][0], timezone="UTC"))
print("last matching bar:", np.datetime_as_string(open_time_index[time_mask][-1], timezone="UTC"))


In [ ]:
sample_indices = np.flatnonzero(time_mask)[:5]
sample_prices = [
    {
        "open_time": np.datetime_as_string(open_time_index[idx], timezone="UTC"),
        "close_time": np.datetime_as_string(close_time_index[idx], timezone="UTC"),
        "open": float(price_ohlcv[idx, 0]),
        "high": float(price_ohlcv[idx, 1]),
        "low": float(price_ohlcv[idx, 2]),
        "close": float(price_ohlcv[idx, 3]),
        "volume": float(price_ohlcv[idx, 4]),
    }
    for idx in sample_indices
]
sample_prices


In [ ]:
signal_probe = [
    {
        "open_time": np.datetime_as_string(open_time_index[idx], timezone="UTC"),
        "dema_row_0": int(dema_signals[0, idx]),
        "ema_row_0": int(ema_signals[0, idx]),
    }
    for idx in range(12)
]
signal_probe


## Notes

- The two `signals.i8.npy` files above are the exact artifact-backed matrices used by the run.
- Each file stores the full indicator matrix for one indicator family on the pinned `1h` timeline.
- The run-specific subset is narrower than the full matrix:
  - `ma.dema`: `source=close`, `window=5..200`
  - `ma.ema`: `source in {high, ohlc4}`, `window=5..200`
- This draft notebook intentionally stops at loading the exact pinned `npy` artifacts and probing timeline alignment.
- Next extension: reconstruct the row mapping for the selected sources/windows from the v2 signal rules/defaults catalog and then slice the relevant rows from each matrix.
